In [0]:
from pyspark.sql.functions import *
import json

In [0]:
spark.range(1).select(date_format(current_timestamp(), "yyyyMMddHHmmssSSS").cast("bigint").alias('partition')).collect()[0]['partition']

In [0]:
for job in (30001, 30002, 30003, 30004, 30005, 30006, 30007, 30008, 30009):
    with open(f"/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/jobs/gold/{job}.json", "r") as job:
        job_details = json.load(job)
        job_details["partition"] = spark.range(1).select(date_format(current_timestamp(), "yyyyMMddHHmmssSSS").cast("bigint").alias('partition')).collect()[0]['partition']
        dbutils.notebook.run("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/ETL/scd_type_1", 500, {"job_parameters" : json.dumps(job_details)})

In [0]:
spark.range(1).select(date_format(current_timestamp(), "yyyyMMddHHmmssSSS").cast("bigint").alias('partition')).collect()[0]['partition']

In [0]:
%sql
create database if not exists ecommerce.ml

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.ml.model_registry (
    -- ── Model Identity ────────────────────────────────────────────────────
    model_name          STRING          NOT NULL,   -- e.g. ecommerce_churn_label_xgboost
    model_type          STRING,                     -- xgboost | ridge | kmeans | als
    target_col          STRING,                     -- churn_label | clv_next_90d_estimate
    source_view         STRING,                     -- gold.vw_customer_features

    -- ── MLflow References ─────────────────────────────────────────────────
    run_id              STRING          NOT NULL,   -- MLflow run ID
    model_uri           STRING,                     -- runs:/<run_id>/xgboost_tuned
    mlflow_version      STRING,                     -- Model Registry version number

    -- ── Model Performance ────────────────────────────────────────────────
    auc_roc             DOUBLE,                     -- primary metric for classifiers
    f1_score            DOUBLE,
    precision_score     DOUBLE,
    recall_score        DOUBLE,
    cv_auc_mean         DOUBLE,                     -- cross-validation AUC mean
    cv_auc_std          DOUBLE,                     -- cross-validation AUC std dev

    -- ── Inference Config ──────────────────────────────────────────────────
    best_threshold      DOUBLE,                     -- optimal classification threshold
    score_col           STRING,                     -- output score column name
    label_col           STRING,                     -- output label column name
    feature_cols        STRING,                     -- JSON array of feature column names
    primary_keys        STRING,                     -- JSON array of primary key columns

    -- ── Job Metadata ──────────────────────────────────────────────────────
    job_id              STRING,                     -- e.g. ML_001
    parent_job_id       STRING,                     -- parent job reference
    partition           STRING,                     -- pipeline partition timestamp

    -- ── Lifecycle ────────────────────────────────────────────────────────
    status              STRING,                     -- production | staging | archived
    registered_at       TIMESTAMP,
    updated_at          TIMESTAMP,
    retired_at          TIMESTAMP,                  -- set when model is archived
    registered_by       STRING                      -- email / service principal

)
USING DELTA
COMMENT 'Model registry table — tracks best MLflow run per model, used by inference pipelines to load correct model version'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
);

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.vw_customer_features AS

WITH reference_date AS (
    SELECT MAX(order_purchase_timestamp) AS ref_date
    FROM ecommerce.silver.vw_cln_ltst_orders
),

rfm_raw AS (
    SELECT
        coh.customer_unique_id,
        coh.customer_state,
        coh.customer_city,

        -- Recency  
        DATEDIFF(r.ref_date, coh.last_order_date)               AS recency_days,

        -- Frequency 
        coh.total_orders                                        AS frequency,

        -- Monetary
        coh.total_spend                                         AS monetary,

        -- Order behaviour 
        coh.avg_order_value,
        coh.max_order_value,
        coh.avg_delivery_days,
        coh.avg_delay_days,
        coh.late_orders,
        coh.late_order_rate,
        coh.delivered_orders,

        -- Payment behaviour
        coh.avg_pct_credit_card,
        coh.installment_orders,
        coh.avg_installments,
        coh.total_freight_paid,
        coh.freight_to_spend_ratio,

        -- Product behaviour
        coh.total_items_purchased,
        coh.avg_items_per_order,
        coh.total_category_touches,

        -- Review behaviour 
        coh.avg_review_score,
        coh.reviews_with_comment,
        coh.low_score_reviews,

        -- Tenure
        coh.customer_tenure_days,
        coh.active_months,
        coh.avg_spend_per_month,
        coh.first_order_date,
        coh.last_order_date,

        r.ref_date,
        coh.total_orders,
        coh.total_spend

    FROM ecommerce.gold.customer_order_history coh
    CROSS JOIN reference_date r
),

rfm_scored AS (
    SELECT
        *,

        -- RFM Quintile Scoring (1=worst, 5=best)
        NTILE(5) OVER (ORDER BY recency_days DESC)              AS R_score,
        NTILE(5) OVER (ORDER BY frequency ASC)                  AS F_score,
        NTILE(5) OVER (ORDER BY monetary ASC)                   AS M_score

    FROM rfm_raw
),

rfm_labeled AS (
    SELECT
        *,

        -- Composite RFM Score
        R_score + F_score + M_score                             AS RFM_score,

        -- RFM Segment Label
        CASE
            WHEN R_score + F_score + M_score >= 12 THEN 'Champions'
            WHEN R_score + F_score + M_score >= 9  THEN 'Loyal Customers'
            WHEN R_score + F_score + M_score >= 7  THEN 'Potential Loyalists'
            WHEN R_score + F_score + M_score >= 5  THEN 'At Risk'
            ELSE                                        'Lost'
        END                                                     AS rfm_segment,

        -- ── Churn Label (FIXED) ───────────────────────────────────────
        -- Multi-order customers: churned if silent for 90+ days
        -- Single-order customers: churned only if 180+ days ago
        --   (gives them fair window to return before being flagged)
        -- Recently acquired (< 90 days): never flagged as churned
        -- Replace churn_label and is_inactive_90d in rfm_labeled CTE:

        -- ── Observation cutoff: customers acquired before this date ──────────────
        -- Use Jan 2018 as cutoff — gives at least 90 days of prediction window
        -- before dataset ends (Aug 2018)

        -- Customers acquired after Jan 2018 are too new → excluded from churn target
        -- by setting churn_label = NULL (model will drop these rows)

        CASE
            -- Too recently acquired — no prediction window → exclude
            WHEN first_order_date >= '2018-01-01'
                THEN NULL

            -- Repeat buyers: churned if silent 90+ days before dataset end
            WHEN total_orders >= 2
            AND DATEDIFF(ref_date, last_order_date) > 90
                THEN 1

            -- Repeat buyers: active within 90 days = not churned
            WHEN total_orders >= 2
            AND DATEDIFF(ref_date, last_order_date) <= 90
                THEN 0

            -- Single-order buyers: churned if 180+ days silent
            -- AND had enough time to return (ordered before Jul 2017)
            WHEN total_orders = 1
            AND first_order_date < '2017-07-01'
            AND DATEDIFF(ref_date, last_order_date) > 180
                THEN 1

            -- Single-order buyers: ordered recently enough → not churned yet
            WHEN total_orders = 1
            AND first_order_date >= '2017-07-01'
                THEN 0

            ELSE 0
        END                                                         AS churn_label,

        -- Original 90-day inactivity — kept as a feature
        CASE
            WHEN DATEDIFF(ref_date, last_order_date) > 90 THEN 1
            ELSE 0
        END                                                         AS is_inactive_90d,

        -- ── CLV Targets ───────────────────────────────────────────────

        -- 1. Raw total spend — no nulls, base CLV
        CAST(total_spend AS DOUBLE)                             AS clv_total_spend,

        -- 2. Spend per active month — normalised across tenures
        CAST(
            ROUND(total_spend / NULLIF(active_months, 0), 2)
        AS DOUBLE)                                              AS clv_per_active_month,

        -- 3. Annualised CLV — tenure-based for repeat buyers,
        --    spend proxy (×4) for single-order customers
        CAST(
            CASE
                WHEN customer_tenure_days > 0
                    THEN ROUND(total_spend / customer_tenure_days * 365, 2)
                ELSE
                    ROUND(total_spend * 4, 2)
            END
        AS DOUBLE)                                              AS annualised_clv,

        -- 4. CLV calculation method flag
        CASE
            WHEN customer_tenure_days > 0 THEN 'tenure_based'
            ELSE                               'spend_proxy'
        END                                                     AS clv_calculation_method,

        -- 5. Next 90 day CLV estimate — ML regression target
        -- Replace clv_next_90d_estimate in rfm_labeled CTE:
        CAST(
            LEAST(
                ROUND(
                    CASE
                        WHEN total_orders >= 2
                        AND DATEDIFF(ref_date, last_order_date) <= 90
                            THEN avg_order_value
                                * (total_orders / NULLIF(customer_tenure_days, 0) * 90)
                        WHEN total_orders >= 2
                            THEN avg_order_value * 0.5
                        ELSE
                            avg_order_value * 0.1
                    END
                , 2),
                500.0   -- cap at 500 BRL (reasonable 90-day CLV ceiling for Olist)
            )
        AS DOUBLE)                                          AS clv_next_90d_estimate,

        -- Breadth of category engagement
        CAST(
            ROUND(total_category_touches / NULLIF(total_orders, 0), 2)
        AS DOUBLE)                                              AS avg_categories_per_order,

        -- Loyalty flag
        CASE
            WHEN total_orders >= 3 THEN 1
            ELSE 0
        END                                                     AS is_repeat_customer

    FROM rfm_scored
)

SELECT
    -- Identity
    customer_unique_id,
    customer_state,
    customer_city,

    -- Core RFM
    recency_days,
    frequency,
    CAST(monetary               AS DOUBLE)                      AS monetary,
    R_score,
    F_score,
    M_score,
    RFM_score,
    rfm_segment,

    -- ML Targets
    churn_label,
    is_inactive_90d,
    clv_total_spend,
    clv_per_active_month,
    annualised_clv,
    clv_calculation_method,
    clv_next_90d_estimate,

    -- Behavioural Features
    CAST(avg_order_value        AS DOUBLE)                      AS avg_order_value,
    CAST(max_order_value        AS DOUBLE)                      AS max_order_value,
    CAST(avg_delivery_days      AS DOUBLE)                      AS avg_delivery_days,
    CAST(avg_delay_days         AS DOUBLE)                      AS avg_delay_days,
    CAST(late_order_rate        AS DOUBLE)                      AS late_order_rate,
    late_orders,
    CAST(avg_review_score       AS DOUBLE)                      AS avg_review_score,
    low_score_reviews,
    reviews_with_comment,

    -- Payment Features
    CAST(avg_pct_credit_card    AS DOUBLE)                      AS avg_pct_credit_card,
    CAST(avg_installments       AS DOUBLE)                      AS avg_installments,
    installment_orders,
    CAST(freight_to_spend_ratio AS DOUBLE)                      AS freight_to_spend_ratio,

    -- Product Features
    total_items_purchased,
    CAST(avg_items_per_order    AS DOUBLE)                      AS avg_items_per_order,
    CAST(avg_categories_per_order AS DOUBLE)                    AS avg_categories_per_order,

    -- Tenure & Activity
    customer_tenure_days,
    active_months,
    CAST(avg_spend_per_month    AS DOUBLE)                      AS avg_spend_per_month,
    is_repeat_customer,
    first_order_date,
    last_order_date

FROM rfm_labeled;

In [0]:
%sql
select customer_unique_id, count(*) from ecommerce.gold.vw_customer_features group by 1 having count(*) > 1

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.vw_product_features AS

WITH product_orders AS (
    SELECT
        i.product_id,
        i.category_english,
        i.product_weight_g,
        i.product_volume_cm3,

        -- Sales velocity 
        COUNT(DISTINCT i.order_id) AS total_orders,
        COUNT(i.order_item_id) AS total_units_sold,
        SUM(i.price) AS total_revenue,
        AVG(i.price) AS avg_price,
        MIN(i.price) AS min_price,
        MAX(i.price) AS max_price,
        STDDEV(i.price) AS price_stddev,

        -- Freight metrics
        AVG(i.freight_value) AS avg_freight,
        AVG(i.freight_pct_of_revenue) AS avg_freight_pct,

        --  Seller spread ─
        COUNT(DISTINCT i.seller_id) AS num_sellers,

        --  High value flag ─
        SUM(i.is_high_value_item) AS high_value_units

    FROM ecommerce.gold.order_items_enriched i
    GROUP BY
        i.product_id,
        i.category_english,
        i.product_weight_g,
        i.product_volume_cm3
),

product_reviews AS (
    SELECT
        i.product_id,
        AVG(r.review_score) AS avg_review_score,
        COUNT(r.review_id) AS total_reviews,
        SUM(r.is_low_score) AS low_score_count,
        SUM(r.has_comment) AS reviews_with_comment
    FROM ecommerce.gold.order_items_enriched i
    LEFT JOIN ecommerce.gold.enriched_reviews r 
      ON i.order_id = r.order_id
    GROUP BY i.product_id
),

category_stats AS (
    -- Category-level benchmarks for relative scoring
    SELECT
        category_english,
        AVG(total_revenue) AS category_avg_revenue,
        AVG(avg_price) AS category_avg_price,
        SUM(total_units_sold) AS category_total_units
    FROM product_orders
    GROUP BY category_english
)

SELECT
    po.product_id,
    po.category_english,
    po.product_weight_g,
    po.product_volume_cm3,

    --  Sales metrics ─
    po.total_orders,
    po.total_units_sold,
    po.total_revenue,
    po.avg_price,
    po.min_price,
    po.max_price,
    po.price_stddev,
    po.num_sellers,

    --  Freight metrics ─
    po.avg_freight,
    po.avg_freight_pct,

    --  Review metrics 
    pr.avg_review_score::decimal(38, 18),
    pr.total_reviews,
    pr.low_score_count,
    pr.reviews_with_comment,

    ROUND(try_divide(pr.low_score_count, pr.total_reviews) * 100, 2)::decimal(38, 18) AS low_score_rate,

    --  Category relative metrics ─
    cs.category_avg_revenue,
    cs.category_total_units,

    ROUND(try_divide(po.total_revenue, cs.category_avg_revenue), 5) AS revenue_vs_category_avg,

    --  Product flags ─
    CASE WHEN po.avg_price >= 200 THEN 1 ELSE 0 END AS is_high_value_product,

    CASE
        WHEN po.total_units_sold >= 50 THEN 'High'
        WHEN po.total_units_sold >= 10 THEN 'Medium'
        ELSE 'Low'
    END AS sales_velocity_tier

FROM product_orders po
LEFT JOIN product_reviews pr 
    ON po.product_id = pr.product_id
LEFT JOIN category_stats cs 
    ON po.category_english = cs.category_english;

In [0]:
%sql
select product_id, count(*) from ecommerce.gold.vw_product_features group by 1 having count(*)> 1

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.vw_business_kpis AS

WITH monthly_orders AS (
    SELECT
        DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS month,
        COUNT(DISTINCT o.order_id) AS total_orders,

        -- Fix: use customer_unique_id for true unique customer count
        COUNT(DISTINCT c.customer_unique_id) AS active_customers,

        SUM(o.is_delivered) AS delivered_orders,
        SUM(o.is_late) AS late_orders,
        AVG(o.actual_delivery_days) AS avg_delivery_days
    FROM ecommerce.gold.orders_enriched  o
    JOIN ecommerce.silver.vw_cln_ltst_customers  c 
        ON o.customer_id = c.customer_id
    GROUP BY DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM')
),

monthly_revenue AS (
    SELECT
        DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS month,
        SUM(p.total_payment_value) AS total_revenue,
        AVG(p.total_payment_value) AS avg_order_value,
        SUM(p.credit_card_value) AS credit_card_revenue,
        SUM(p.boleto_value) AS boleto_revenue,
        SUM(p.voucher_value) AS voucher_revenue,
        SUM(p.is_installment) AS installment_orders
    FROM ecommerce.gold.orders_enriched   o
    LEFT JOIN ecommerce.gold.payments_agg p 
        ON o.order_id = p.order_id
    GROUP BY DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM')
),

monthly_reviews AS (
    SELECT
        DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS month,
        AVG(r.review_score) AS avg_review_score,
        SUM(r.is_low_score) AS low_score_reviews,
        COUNT(r.review_id) AS total_reviews
    FROM ecommerce.gold.orders_enriched o
    LEFT JOIN ecommerce.gold.vw_enriched_reviews r 
        ON o.order_id = r.order_id
    GROUP BY DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM')
),

true_first_orders AS (
    -- Fix: derive first order date per real customer using customer_unique_id
    SELECT
        c.customer_unique_id,
        MIN(o.order_purchase_timestamp) AS first_order_date
    FROM ecommerce.silver.vw_cln_ltst_orders o
    JOIN ecommerce.silver.vw_cln_ltst_customers c 
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),

monthly_new_customers AS (
    -- Fix: a customer is "new" only in the month of their very first order
    SELECT
        DATE_FORMAT(first_order_date, 'yyyy-MM') AS month,
        COUNT(customer_unique_id) AS new_customers
    FROM true_first_orders
    GROUP BY DATE_FORMAT(first_order_date, 'yyyy-MM')
)
SELECT
    mo.month,

    -- ── Volume KPIs ───────────────────────────────────────────────
    mo.total_orders,
    mo.active_customers,
    COALESCE(mn.new_customers, 0) AS new_customers,

    -- Fix: returning = total active minus genuinely new this month
    mo.active_customers - COALESCE(mn.new_customers, 0) AS returning_customers,

    -- ── Revenue KPIs ──────────────────────────────────────────────
    mr.total_revenue,
    mr.avg_order_value,
    ROUND(try_divide(mr.total_revenue, mo.active_customers), 5) AS revenue_per_customer,

    -- ── Revenue MoM Growth ────────────────────────────────────────
    ROUND(try_divide(
        (mr.total_revenue - LAG(mr.total_revenue) OVER (ORDER BY mo.month)),
        LAG(mr.total_revenue) OVER (ORDER BY mo.month)) * 100
    , 5) AS revenue_mom_growth_pct,

    -- ── Payment Mix ───────────────────────────────────────────────
    mr.credit_card_revenue,
    mr.boleto_revenue,
    mr.voucher_revenue,
    mr.installment_orders,

    -- ── Operations KPIs ───────────────────────────────────────────
    mo.delivered_orders,
    mo.late_orders,
    ROUND(try_divide(mo.late_orders, mo.delivered_orders) * 100, 5) AS late_delivery_rate_pct,
    mo.avg_delivery_days,

    -- ── Quality KPIs ──────────────────────────────────────────────
    rv.avg_review_score,
    rv.low_score_reviews,
    rv.total_reviews,
    ROUND(try_divide(rv.low_score_reviews, rv.total_reviews) * 100, 5) AS low_score_rate_pct,

    -- ── Customer Growth MoM ───────────────────────────────────────
    ROUND(try_divide(
        (mo.active_customers - LAG(mo.active_customers) OVER (ORDER BY mo.month)),
        LAG(mo.active_customers) OVER (ORDER BY mo.month)) * 100
    , 5) AS customer_growth_mom_pct

FROM monthly_orders mo
LEFT JOIN monthly_revenue mr 
    ON mo.month = mr.month
LEFT JOIN monthly_reviews rv 
    ON mo.month = rv.month
LEFT JOIN monthly_new_customers mn 
    ON mo.month = mn.month
where mo.month NOT IN ('2016-09', '2016-12', '2018-09', '2018-10')

In [0]:
%sql
select * from ecommerce.gold.vw_business_kpis

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.vw_cohort_analysis AS

WITH 
filtered_orders AS (
    -- Fix: exclude partial edge months with unreliable data
    SELECT *
    FROM ecommerce.silver.vw_cln_ltst_orders
    WHERE DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') NOT IN ('2016-09', '2016-12', '2018-09', '2018-10')
),

customer_cohorts AS (
    -- Use customer_unique_id as the true customer identifier
    -- First order date = acquisition date
    SELECT
        c.customer_unique_id,
        DATE_FORMAT(MIN(o.order_purchase_timestamp), 'yyyy-MM') AS cohort_month,
        MIN(o.order_purchase_timestamp) AS first_order_date
    FROM filtered_orders o
    JOIN ecommerce.silver.vw_cln_ltst_customers c 
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),

customer_activity AS (
    -- All months a real customer placed any order
    -- Deduped at customer_unique_id + activity_month grain
    SELECT DISTINCT
        c.customer_unique_id,
        DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS activity_month
    FROM filtered_orders o
    JOIN ecommerce.silver.vw_cln_ltst_customers c 
        ON o.customer_id = c.customer_id
),

cohort_activity AS (
    SELECT
        cc.cohort_month,
        ca.activity_month,

        -- Months since acquisition (0 = acquisition month itself)
        (
          (YEAR(TO_DATE(CONCAT(ca.activity_month, '-01'))) - YEAR(TO_DATE(CONCAT(cc.cohort_month, '-01')))) * 12
          + (MONTH(TO_DATE(CONCAT(ca.activity_month, '-01'))) - MONTH(TO_DATE(CONCAT(cc.cohort_month, '-01'))))
        ) AS months_since_acquisition,
        COUNT(DISTINCT cc.customer_unique_id) AS active_customers

    FROM customer_cohorts   cc
    LEFT JOIN customer_activity ca 
        ON cc.customer_unique_id = ca.customer_unique_id
    GROUP BY
        1, 2, 3
),

cohort_sizes AS (
    SELECT
        cohort_month,
        COUNT(customer_unique_id) AS cohort_size
    FROM customer_cohorts
    GROUP BY cohort_month
)

SELECT
    ca.cohort_month,
    cs.cohort_size,
    ca.months_since_acquisition,
    ca.activity_month,
    ca.active_customers,

    -- Retention rate % vs original cohort size
    ROUND(try_divide(ca.active_customers, cs.cohort_size) * 100, 5) AS retention_rate_pct,

    -- Customers lost compared to previous month in same cohort
    ca.active_customers - LAG(ca.active_customers) OVER (PARTITION BY ca.cohort_month ORDER BY ca.months_since_acquisition) AS customers_lost_vs_prior_month

FROM cohort_activity ca
LEFT JOIN cohort_sizes cs 
    ON ca.cohort_month = cs.cohort_month
ORDER BY ca.cohort_month, ca.months_since_acquisition;

In [0]:
%sql
select * from ecommerce.gold.vw_cohort_analysis